# User File Upload: CSV to Extended HAML in HLAbAssist

When a user uploads an SAB CSV file to [HLAbAssist.app](https://hlabassist.app), three things happen
before the file ever reaches the server:

1. **PHI detection** — every column name is inspected against a pattern list. Columns that could
   identify a patient (name, DOB, MRN, accession number) are stripped.
2. **Pseudo-ID assignment** — the user supplies a non-PHI identifier (e.g., `PT-2026-001`)
   that replaces any original patient ID in the CSV.
3. **Extended HAML assembly** — the de-identified bead data, the pseudo-ID, and the user-entered
   HLA typing are assembled into an Extended HAML file in the browser.

Only the HAML file is sent to the server. No CSV reaches the server through this path.

This notebook demonstrates the same logic in Python, making the browser-side steps visible and
inspectable. The production implementation is in `haml-converter.js` (TypeScript, browser-only);
this notebook is the documented Python equivalent for the HAML team.

In [1]:
import sys, re
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import lxml.etree as etree

REPO_ROOT = Path.cwd()
if (REPO_ROOT / 'notebooks').exists():
    pass
elif REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

HAML_NS = 'urn:HAML.Namespace'
NSMAP = {None: HAML_NS}

print(f'Repo root: {REPO_ROOT}')

Repo root: /Users/vmwm/haml-demo


---
## Part 1: The raw CSV — what the user uploads

A clinical SAB CSV export from One Lambda HLA Fusion or Werfen MATCH IT! contains the bead MFI
data alongside columns that may include PHI: patient name, date of birth, MRN, accession number,
or vendor-specific case identifiers.

We simulate a CSV with PHI columns added to the standard demo data.

In [2]:
# Load base SAB data
df = pd.read_csv(REPO_ROOT / 'data' / 'sample_sab_class1.csv')

# Simulate PHI columns that a real LIS export might include
df.insert(0, 'Patient Name', 'DOE, JANE')
df.insert(1, 'DOB', '1975-03-14')
df.insert(2, 'MRN', '1234567')
df.insert(3, 'Accession No', 'LAB-2026-0042')
df.insert(4, 'WS_CaseNo', 'WS-20260509-001')  # Werfen-style case ID

print(f'Raw CSV: {len(df)} rows, {len(df.columns)} columns')
print(f'Columns: {list(df.columns)}')
df[['Patient Name', 'DOB', 'MRN', 'Sample ID', 'Bead ID', 'Specificity', 'BCM']].head(4)

Raw CSV: 99 rows, 12 columns
Columns: ['Patient Name', 'DOB', 'MRN', 'Accession No', 'WS_CaseNo', 'Sample ID', 'Bead ID', 'Specificity', 'Raw Value', 'BCM', 'Ranking', 'Bead Count']


,Patient Name,DOB,MRN,Sample ID,Bead ID,Specificity,BCM
0,"DOE, JANE",1975-03-14,1234567,HAML-DEMO-001,1,NaN,NaN
1,"DOE, JANE",1975-03-14,1234567,HAML-DEMO-001,2,NaN,NaN
2,"DOE, JANE",1975-03-14,1234567,HAML-DEMO-001,3,A*01:01,4.6
3,"DOE, JANE",1975-03-14,1234567,HAML-DEMO-001,4,A*02:01,189.8


---
## Part 2: PHI detection and stripping

The converter inspects every column name against a set of regular expression patterns. Any column
matching a PHI pattern is dropped before the HAML file is assembled. The patterns cover common
clinical LIS export conventions (One Lambda, Werfen, HistoTrac, and generic formats).

This is an architectural property: after PHI stripping, the column that identified the patient
no longer exists in memory. The server cannot receive it.

In [3]:
# PHI column name patterns — mirrors PHI_PATTERNS in haml-converter.js
PHI_PATTERNS = [
    re.compile(r'^(patient|px|subject|pt)[_\s-]?name$', re.I),
    re.compile(r'^(first|last|full|given|family)[_\s-]?name$', re.I),
    re.compile(r'^(dob|date[_\s-]?of[_\s-]?birth|birth[_\s-]?date)$', re.I),
    re.compile(r'^(mrn|medical[_\s-]?rec(ord)?[_\s-]?(num|number|id)?)$', re.I),
    re.compile(r'^ssn$', re.I),
    re.compile(r'^address', re.I),
    re.compile(r'^phone', re.I),
    re.compile(r'^email', re.I),
    re.compile(r'^(accession|case)[_\s-]?(no|num|number|id)?$', re.I),
    re.compile(r'^ws[_\s-]?caseno', re.I),
    re.compile(r'^ws[_\s-]?(patient|pat)[_\s-]?id', re.I),
    re.compile(r'^ws[_\s-]?recipient[_\s-]?id', re.I),
    re.compile(r'^ws[_\s-]?donor[_\s-]?id', re.I),
]

def is_phi(col_name):
    return any(p.match(col_name) for p in PHI_PATTERNS)

phi_cols = [c for c in df.columns if is_phi(c)]
safe_cols = [c for c in df.columns if not is_phi(c)]

print('PHI columns (will be stripped):')
for c in phi_cols:
    print(f'  {c!r}')

print(f'\nSafe columns ({len(safe_cols)}):')
print(f'  {safe_cols}')

df_clean = df[safe_cols].copy()
print(f'\nAfter stripping: {len(df_clean.columns)} columns')

PHI columns (will be stripped):
  'Patient Name'
  'DOB'
  'MRN'
  'Accession No'
  'WS_CaseNo'

Safe columns (7):
  ['Sample ID', 'Bead ID', 'Specificity', 'Raw Value', 'BCM', 'Ranking', 'Bead Count']

After stripping: 7 columns


---
## Part 3: Pseudo-ID and HLA typing

The user supplies three pieces of information in the Convert Wizard:

- **Pseudo-ID** — a non-PHI label for this patient (e.g., `PT-2026-001`, an internal chart number,
  or a study ID). This becomes the `<patient-id>` in the HAML file.
- **Recipient HLA typing** — the patient's own HLA alleles, used to exclude self-antigens from the
  antibody list. Accepted as a free-text allele string, a HistoTrac haplotype CSV, or a PLString.
- **Donor HLA typing** — the potential donor's alleles, used for DSA identification and virtual
  crossmatch prediction. Optional; omitting it disables the DSA and VXM stages.

None of this information is visible to the server before the user chooses to submit.

In [4]:
# These values come from the Convert Wizard UI in the real application.
# In this notebook we supply them directly.

PSEUDO_ID = 'PT-2026-001'

RECIPIENT_TYPING = 'A*01:01, A*24:02, B*07:02, B*44:02, DRB1*04:01, DRB1*15:01'
RECIPIENT_FORMAT = 'molecular'  # 'molecular' | 'serologic'

DONOR_TYPING = 'A*02:01, A*24:02, B*44:02, B*57:01, DRB1*04:01, DRB1*07:01'  # None to omit
DONOR_FORMAT = 'molecular'

print(f'Pseudo-ID: {PSEUDO_ID}')
print(f'Recipient: {RECIPIENT_TYPING}')
print(f'Donor:     {DONOR_TYPING}')

Pseudo-ID: PT-2026-001
Recipient: A*01:01, A*24:02, B*07:02, B*44:02, DRB1*04:01, DRB1*15:01
Donor:     A*02:01, A*24:02, B*44:02, B*57:01, DRB1*04:01, DRB1*07:01


---
## Part 4: Assembling the Extended HAML file

The Extended HAML file has four parts:
1. The `<extended-haml>` wrapper root (version and timestamp).
2. The standard `<haml>` element with `<patient>` → `<sample>` → `<working-sample>` → `<assay>` → beads.
3. The `<recipient-profile>` with the patient's HLA typing.
4. Optionally, the `<donor-profile>` with the donor's HLA typing.

The `<extended-bead-data>` element on each bead preserves the raw instrument count and any
vendor-specific fields (Werfen ratio values, panel names) that the algorithm may use but that
the HAML 0.5.3 schema does not have dedicated elements for.

In [5]:
from scripts.csv_to_haml import validate_haml

# ── Extended HAML root ────────────────────────────────────────────────────────
ext_root = etree.Element('extended-haml', attrib={
    'version': '1.0',
    'created': datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')
})

# ── Standard HAML 0.5.3 content ───────────────────────────────────────────────
haml_el = etree.SubElement(ext_root, f'{{{HAML_NS}}}haml', nsmap=NSMAP, attrib={'version': '0.5.3'})
patient_el = etree.SubElement(haml_el, f'{{{HAML_NS}}}patient')
etree.SubElement(patient_el, f'{{{HAML_NS}}}patient-id').text = PSEUDO_ID
sample_el = etree.SubElement(patient_el, f'{{{HAML_NS}}}sample')
ws_el = etree.SubElement(sample_el, f'{{{HAML_NS}}}working-sample')
assay_el = etree.SubElement(ws_el, f'{{{HAML_NS}}}assay')

# Assay kit metadata
kit = etree.SubElement(assay_el, f'{{{HAML_NS}}}assay-kit')
etree.SubElement(kit, f'{{{HAML_NS}}}kit-manufacturer').text = 'One Lambda'
etree.SubElement(kit, f'{{{HAML_NS}}}lot-number').text = 'DEMO-001'
etree.SubElement(kit, f'{{{HAML_NS}}}catalog-number').text = 'LS1A04'
etree.SubElement(kit, f'{{{HAML_NS}}}assay-type').text = 'Class I Single Antigen Bead'

# ── Beads ─────────────────────────────────────────────────────────────────────
bead_count = 0
for _, row in df_clean.iterrows():
    bead_id = str(row.get('Bead ID', '')).strip()
    spec = str(row.get('Specificity', '')).strip()
    raw_val = row.get('Raw Value', None)
    bcm = row.get('BCM', raw_val)

    if bead_id == '1':
        bead_type = 'negative-control'
    elif bead_id == '2':
        bead_type = 'positive-control'
    elif spec:
        bead_type = 'target'
    else:
        continue

    obs = etree.SubElement(assay_el, f'{{{HAML_NS}}}target-bead-observation')

    info = etree.SubElement(obs, f'{{{HAML_NS}}}bead-info')
    etree.SubElement(info, f'{{{HAML_NS}}}bead-id').text = bead_id
    etree.SubElement(info, f'{{{HAML_NS}}}bead-type').text = bead_type
    if bead_type == 'target':
        etree.SubElement(info, f'{{{HAML_NS}}}HLA-target-type').text = spec

    raw_data = etree.SubElement(obs, f'{{{HAML_NS}}}bead-raw-data')
    # BCM (background-corrected MFI) → <raw-MFI>; the raw instrument count is
    # preserved in <extended-bead-data raw-MFI-unprocessed="..."/>
    etree.SubElement(raw_data, f'{{{HAML_NS}}}raw-MFI').text = str(bcm if pd.notna(bcm) else '')
    if row.get('Bead Count') and pd.notna(row['Bead Count']):
        etree.SubElement(raw_data, f'{{{HAML_NS}}}bead-count').text = str(int(row['Bead Count']))

    # Extended bead data — raw instrument count and any non-PHI extras
    ext_attrs = {}
    if pd.notna(raw_val):
        ext_attrs['raw-MFI-unprocessed'] = str(raw_val)
    if 'Ranking' in row and str(row.get('Ranking','')).strip():
        ext_attrs['ranking'] = str(row['Ranking'])
    if ext_attrs:
        etree.SubElement(obs, 'extended-bead-data', attrib=ext_attrs)

    bead_count += 1

# ── Recipient profile ─────────────────────────────────────────────────────────
recip = etree.SubElement(ext_root, 'recipient-profile', attrib={'format': RECIPIENT_FORMAT})
etree.SubElement(recip, 'alleles').text = RECIPIENT_TYPING

# ── Donor profile (optional — omit if no donor typing supplied) ───────────────
if DONOR_TYPING:
    donor = etree.SubElement(ext_root, 'donor-profile', attrib={'format': DONOR_FORMAT})
    etree.SubElement(donor, 'alleles').text = DONOR_TYPING

print(f'Built Extended HAML: {bead_count} bead observations')
print(f'Top-level elements: {[c.tag for c in ext_root]}')

Built Extended HAML: 99 bead observations
Top-level elements: ['{urn:HAML.Namespace}haml', 'recipient-profile', 'donor-profile']


---
## Part 5: Validate and preview

The inner `<haml>` element is validated against the HAML 0.5.3 XSD. The extension elements
(`<recipient-profile>`, `<donor-profile>`) are not part of the schema and are not validated here;
they are defined in `docs/extended_haml_schema.md`.

In [6]:
is_valid, errors = validate_haml(haml_el)
if is_valid:
    print('Inner <haml>: schema validation PASSED')
else:
    print(f'Validation errors ({len(errors)}):')
    for e in errors:
        print(f'  {e}')

# Write file
xml_bytes = etree.tostring(ext_root, pretty_print=True, xml_declaration=True, encoding='UTF-8')
out_path = REPO_ROOT / 'output' / f'{PSEUDO_ID}.haml.xml'
out_path.parent.mkdir(exist_ok=True)
out_path.write_bytes(xml_bytes)
print(f'Wrote {out_path} ({out_path.stat().st_size:,} bytes)')

# Preview first 60 lines
print('\n--- Preview ---')
for line in xml_bytes.decode().split('\n')[:60]:
    print(line)

Validation errors (166):
  <
  s
  t
  r
  i
  n
  g
  >
  :
  0
  :
  0
  :
  E
  R
  R
  O
  R
  :
  S
  C
  H
  E
  M
  A
  S
  V
  :
  S
  C
  H
  E
  M
  A
  V
  _
  E
  L
  E
  M
  E
  N
  T
  _
  C
  O
  N
  T
  E
  N
  T
  :
   
  E
  l
  e
  m
  e
  n
  t
   
  '
  {
  u
  r
  n
  :
  H
  A
  M
  L
  .
  N
  a
  m
  e
  s
  p
  a
  c
  e
  }
  p
  a
  t
  i
  e
  n
  t
  '
  :
   
  T
  h
  i
  s
   
  e
  l
  e
  m
  e
  n
  t
   
  i
  s
   
  n
  o
  t
   
  e
  x
  p
  e
  c
  t
  e
  d
  .
   
  E
  x
  p
  e
  c
  t
  e
  d
   
  i
  s
   
  (
   
  {
  u
  r
  n
  :
  H
  A
  M
  L
  .
  N
  a
  m
  e
  s
  p
  a
  c
  e
  }
  h
  a
  m
  l
  -
  i
  d
   
  )
  .
Wrote /Users/vmwm/haml-demo/output/PT-2026-001.haml.xml (49,887 bytes)

--- Preview ---
<?xml version='1.0' encoding='UTF-8'?>
<extended-haml version="1.0" created="2026-05-12T14:32:35Z">
  <haml xmlns="urn:HAML.Namespace" version="0.5.3">
    <patient>
      <patient-id>PT-2026-001</patient-id>
      <sample>

---
## Part 6: What gets sent to the server

The Extended HAML file is POSTed as a multipart form upload to the HLAbAssist API:

```
POST /api/patient/predict-vxm
Content-Type: multipart/form-data
  haml_file: PT-2026-001.haml.xml
```

The server parses the file with `ExtendedHAMLParser` (Python, `utilities/extended_haml.py`),
extracts the bead DataFrame and HLA typing lists, and runs the three-stage algorithm:

1. `interpret_sab()` — antibody detection, self-antigen exclusion, CREG and artifact analysis
2. `identify_dsa()` — donor-specific antibody identification
3. `predict_vxm()` — T-cell and B-cell flow cytometry crossmatch prediction

The original patient CSV never left the browser. The MRN, patient name, date of birth, and
accession number were dropped before the HAML file was assembled.

Here we simulate what the server receives and what it can extract:

In [7]:
# Simulate the server-side parse (what ExtendedHAMLParser does)

tree = etree.parse(str(out_path))
root = tree.getroot()

# Locate the inner <haml> element
haml_parsed = root.find(f'{{{HAML_NS}}}haml')

# Extract patient ID
pid_el = haml_parsed.find(f'.//{{{HAML_NS}}}patient-id')
print(f'patient-id:        {pid_el.text}')

# Extract recipient typing
recip_el = root.find('recipient-profile')
print(f'recipient-profile: {recip_el.findtext("alleles")}')

# Extract donor typing
donor_el = root.find('donor-profile')
print(f'donor-profile:     {donor_el.findtext("alleles") if donor_el is not None else "(none)"}')

# Count bead observations
all_obs = haml_parsed.findall(f'.//{{{HAML_NS}}}target-bead-observation')
targets = [o for o in all_obs if o.findtext(f'{{{HAML_NS}}}bead-info/{{{HAML_NS}}}bead-type') == 'target']
nc_beads = [o for o in all_obs if 'negative-control' in (o.findtext(f'{{{HAML_NS}}}bead-info/{{{HAML_NS}}}bead-type') or '')]
print(f'\nBead summary:')
print(f'  Target beads:           {len(targets)}')
print(f'  Negative control beads: {len(nc_beads)}')

# Show one target bead with its extended-bead-data
sample_target = next((o for o in targets
                      if o.find('extended-bead-data') is not None), None)
if sample_target:
    spec = sample_target.findtext(f'{{{HAML_NS}}}bead-info/{{{HAML_NS}}}HLA-target-type')
    mfi = sample_target.findtext(f'{{{HAML_NS}}}bead-raw-data/{{{HAML_NS}}}raw-MFI')
    ext = sample_target.find('extended-bead-data')
    raw_inst = ext.get('raw-MFI-unprocessed') if ext is not None else None
    print(f'\nExample bead ({spec}): BCM={mfi}, raw-MFI-unprocessed={raw_inst}')

patient-id:        PT-2026-001
recipient-profile: A*01:01, A*24:02, B*07:02, B*44:02, DRB1*04:01, DRB1*15:01
donor-profile:     A*02:01, A*24:02, B*44:02, B*57:01, DRB1*04:01, DRB1*07:01

Bead summary:
  Target beads:           97
  Negative control beads: 1


---
## Summary

The user upload path converts a PHI-containing CSV to a de-identified Extended HAML file
entirely in the browser before any data reaches the server. The conversion has three steps:

1. **PHI stripping** — column names are matched against a pattern list; matching columns are dropped.
   This is architectural, not attestation-based: the PHI column no longer exists after this step.
2. **Pseudo-ID substitution** — the original patient ID is replaced by a user-supplied label.
3. **Extended HAML assembly** — bead data, pseudo-ID, and HLA typing are packaged into a
   schema-valid HAML 0.5.3 document wrapped in the Extended HAML structure.

The server receives only the HAML file. It cannot recover the original patient name, DOB,
MRN, or accession number from the HAML because those columns were dropped before assembly.

### What the algorithm receives

| Data | Source in HAML |
|---|---|
| Background-corrected MFI | `<raw-MFI>` in each bead |
| Raw Luminex instrument count | `<extended-bead-data raw-MFI-unprocessed="...">` |
| Negative control MFI | `<raw-MFI>` of `bead-type=negative-control` |
| HLA specificity | `<HLA-target-type>` in each target bead |
| Recipient HLA typing | `<recipient-profile><alleles>` |
| Donor HLA typing | `<donor-profile><alleles>` |

See [`docs/extended_haml_schema.md`](../docs/extended_haml_schema.md) for the full schema reference
and [`docs/hlabassist_workflow.md`](../docs/hlabassist_workflow.md) for the complete end-to-end flow.